In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
cineiq_df = pd.read_csv("../CineIq_Data/modify/cineiq_metadata.csv")

In [3]:
cineiq_df.head()

,movieId,title,overview,genres,keywords
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,jealousy toy boy friendship friends rivalry bo...
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,board game disappearance based on children's b...
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Romance Comedy,fishing best friend duringcreditsstinger old men
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance,based on novel interracial relationship single...
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...


In [4]:
cineiq_df["soup"] = (
    cineiq_df["overview"] + " " +
    (cineiq_df["genres"] + " ") * 2 +
    (cineiq_df["keywords"] + " ") * 3
)
cineiq_df = cineiq_df.fillna("")

In [5]:
cineiq_df[["title","soup"]].head()

,title,soup
0,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji (1995),When siblings Judy and Peter discover an encha...
2,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...
3,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...


In [6]:
tfidf = TfidfVectorizer(stop_words="english", min_df=4,max_features=20000)
tfidf_matrix = tfidf.fit_transform(cineiq_df["soup"])

In [7]:
tfidf_matrix.shape

(43963, 20000)

In [8]:
def search_movie(query, df):
    matches = df[df["title"].str.lower().str.contains(query.lower(),na=False)]
    if matches.empty:
        return "No movies found."
    return matches[["movieId","title"]]

In [9]:
search_movie("Titanic",cineiq_df)

,movieId,title
1645,1721,Titanic (1997)
2053,2157,"Chambermaid on the Titanic, The (Femme de cham..."
3291,3403,Raise the Titanic (1980)
3292,3404,Titanic (1953)
4744,4864,Titanica (1992)
15668,82602,Titanic Town (1998)
23245,118916,Titanic (1996)
31066,143539,Titanic at 100: Mystery Solved (2012)
41795,170975,The Ten Lives of Titanics the Cat (2007)
41823,171037,Drain the Titanic (2016)


In [10]:
def get_recommendations_by_index(target_index, tfidf_matrix, df, n=10):
    movie_vector = tfidf_matrix[target_index]
    sim_scores = cosine_similarity(movie_vector, tfidf_matrix).flatten()

    sorted_indices = sim_scores.argsort()[::-1]
    top_indices = sorted_indices[1:n+1]

    recommendations = df.iloc[top_indices][["title"]].copy()
    recommendations["similarity_score"] = sim_scores[top_indices]

    return recommendations

In [11]:
get_recommendations_by_index(23245, tfidf_matrix, cineiq_df)

,title,similarity_score
1645,Titanic (1997),0.406008
3293,"Night to Remember, A (1958)",0.401617
3292,Titanic (1953),0.370848
1910,"Poseidon Adventure, The (1972)",0.329744
4719,Death Ship (1980),0.311177
23814,Carry on Cruising (1962),0.300024
3105,Voyage of the Damned (1976),0.292240
4744,Titanica (1992),0.289077
42769,Titanic: The Final Word with James Cameron (2012),0.288479
2584,"Legend of 1900, The (a.k.a. The Legend of the ...",0.284228


In [12]:
search_movie("spider-man", cineiq_df)

,movieId,title
5228,5349,Spider-Man (2002)
7916,8636,Spider-Man 2 (2004)
11557,52722,Spider-Man 3 (2007)
14517,76709,Spider-Man: The Ultimate Villain Showdown (2002)
18167,95510,"Amazing Spider-Man, The (2012)"
21253,110553,The Amazing Spider-Man 2 (2014)


In [13]:
get_recommendations_by_index(5228, tfidf_matrix, cineiq_df)

,title,similarity_score
11557,Spider-Man 3 (2007),0.373508
26159,Marvel One-Shot: Item 47 (2012),0.309993
3675,X-Men (2000),0.300300
24280,Avengers: Age of Ultron (2015),0.297967
17024,"Avengers, The (2012)",0.286024
14618,Iron Man 2 (2010),0.281340
4517,"Punisher, The (1989)",0.274822
39677,While You Were Fighting: A Thor Mockumentary (...,0.273731
24289,Doctor Strange (2016),0.268844
24283,Ant-Man (2015),0.268152
